# SPARC Example 03: The Omega Kinematic Correction (EPS Research)

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

This is the core EPS Research result. The omega kinematic correction is derived
from boundary points only — no free parameters, no mass modeling:

    V_adj = V_obs - R * omega
    omega = (V2/R2 - V1/R1) * (R1/R2)^(3/2)   [rad/Gyr]

where (R1, V1) = innermost ring, (R2, V2) = outermost ring.
Note: 1 rad/Gyr = 1.022 km/s/kpc.

**Published result:** omega = 4.69 rad/Gyr for DDO161
**Reference:** Flynn & Cannaliato (2025), DOI: 10.3389/fspas.2025.1680387

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

g = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
d = g['data']

R     = np.array([p['Rad']   for p in d])
Vobs  = np.array([p['Vobs']  for p in d])
errV  = np.array([p['errV']  for p in d])
Vgas  = np.array([p['Vgas']  for p in d])
Vdisk = np.array([p['Vdisk'] for p in d])
Vbul  = np.array([p['Vbul']  for p in d])

# Baryonic velocity
Vbar = np.where(Vgas < 0,
                -np.sqrt(Vgas**2 + Vdisk**2 + Vbul**2),
                 np.sqrt(Vgas**2 + Vdisk**2 + Vbul**2))

# Omega from boundary points
R1, V1 = R[0],  Vobs[0]
R2, V2 = R[-1], Vobs[-1]
omega  = (V2/R2 - V1/R1) * (R1/R2)**1.5
V_adj  = Vobs - R * omega

# Keplerian baseline
GM       = V2**2 * R2
V_kepler = np.sqrt(GM / R)

# RMSE
rmse_omega  = np.sqrt(np.mean((V_adj  - Vbar)**2))
rmse_kepler = np.sqrt(np.mean((V_kepler - Vbar)**2))

print(f"Computed omega  = {omega:.3f} rad/Gyr")
print(f"Published omega = 4.69 rad/Gyr (Flynn & Cannaliato 2025)")
print(f"RMSE (omega vs Vbar):   {rmse_omega:.2f} km/s")
print(f"RMSE (Kepler vs Vbar):  {rmse_kepler:.2f} km/s")
print(f"Improvement:            {rmse_kepler - rmse_omega:.2f} km/s")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(R, Vobs, yerr=errV, fmt='o', color='#1f77b4',
            capsize=4, markersize=6, label=r'$V_{\rm obs}$ (SPARC)', zorder=5)
ax.plot(R, Vbar,     's-', color='#d62728', linewidth=1.5,
        label=r'$V_{\rm bar}$ ($\Upsilon=1$)')
ax.plot(R, V_adj,    '^-', color='#2ca02c', linewidth=1.8,
        label=fr'$V_{{\rm obs}} - R\omega$  ($\omega={omega:.2f}$ rad/Gyr)')
ax.plot(R, V_kepler, '--', color='#ff7f0e', linewidth=1.2,
        label='Keplerian baseline')
ax.set_xlabel('Radius (kpc)', fontsize=12)
ax.set_ylabel('Velocity (km/s)', fontsize=12)
ax.set_title('DDO161 — Omega Kinematic Correction\n'
             'Flynn & Cannaliato (2025) | DOI: 10.3389/fspas.2025.1680387', fontsize=10)
ax.legend(fontsize=9)
ax.set_xlim(0, R.max() * 1.15)
ax.set_ylim(0, Vobs.max() * 1.35)
ax.text(0.97, 0.08,
        f'RMSE omega: {rmse_omega:.1f} km/s\nRMSE Kepler: {rmse_kepler:.1f} km/s',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
plt.tight_layout()
plt.savefig('ex03_ddo161_omega.png', dpi=150, bbox_inches='tight')
plt.show()